In [0]:
from pyspark.sql import functions as F

CATALOG = "olist_lakehouse"
SCHEMA = "bronze"
BASE_PATH = "/Volumes/olist_lakehouse/landing/olist/historical"


DEFAULT_OPTIONS = {
    "header": "true",
    "inferSchema": "false",  # lets keep every column type as string
}

TABLES = [
    {"file": "olist_orders_dataset.csv",              "table": "orders",               "comment": "Pedidos (cabeçalho do pedido)"},
    {"file": "olist_order_items_dataset.csv",         "table": "order_items",          "comment": "Itens de cada pedido"},
    {"file": "olist_order_payments_dataset.csv",      "table": "order_payments",       "comment": "Pagamentos de cada pedido"},
    {"file": "olist_order_reviews_dataset.csv",       "table": "order_reviews",        "comment": "Avaliações dos clientes",
     "options": {"multiLine": "true", "escape": '"'}},
    {"file": "olist_customers_dataset.csv",           "table": "customers",            "comment": "Clientes (um customer_id por pedido)"},
    {"file": "olist_products_dataset.csv",            "table": "products",             "comment": "Catálogo de produtos"},
    {"file": "olist_sellers_dataset.csv",             "table": "sellers",              "comment": "Vendedores"},
    {"file": "olist_geolocation_dataset.csv",         "table": "geolocation",          "comment": "Coordenadas por prefixo de CEP"},
    {"file": "product_category_name_translation.csv", "table": "category_translation", "comment": "Tradução das categorias PT para EN"},
]

In [0]:
def ingest_csv_to_bronze(file: str, table: str, comment: str, options: dict | None = None) -> int:
    """Reads a csv file and loads its data into a delta table in bronze layer."""
    full_table_name = f"{CATALOG}.{SCHEMA}.{table}"
    read_options = {**DEFAULT_OPTIONS, **(options or {})}

    df = (
        spark.read
        .format("csv")
        .options(**read_options)
        .load(f"{BASE_PATH}/{file}")
        .select(
            "*",
            F.col("_metadata.file_path").alias("_source_file"),
            F.current_timestamp().alias("_ingested_at"),
        )
    )

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(full_table_name)
    )

    spark.sql(f"COMMENT ON TABLE {full_table_name} IS '{comment}'")

    return spark.table(full_table_name).count()

In [0]:
for cfg in TABLES:
    rows = ingest_csv_to_bronze(
        file=cfg["file"],
        table=cfg["table"],
        comment=cfg["comment"],
        options=cfg.get("options"),
    )
    print(f"{CATALOG}.{SCHEMA}.{cfg['table']:<22} {rows:>10,} rows")

In [0]:
%sql
-- #checking if tables were created

show tables from olist_lakehouse.bronze;


In [0]:
%sql
-- #checking if tables were populated

-- select count(*) from olist_lakehouse.bronze.customers;


-- DESCRIBE TABLE olist_lakehouse.bronze.orders;

-- DESCRIBE DETAIL olist_lakehouse.bronze.orders

DESCRIBE HISTORY olist_lakehouse.bronze.orders